In [1]:
!pip install -q "transformers==4.46.3" accelerate bitsandbytes torch

In [2]:
# [REDACTED_HF_TOKEN]
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "inceptionai/jais-family-6p7b-chat"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

In [114]:
# 3) Jais's native chat template — REQUIRED. jais-chat models are fine-tune
# specifically on this Instruction/Human/AI format. Feeding raw, un-wrapped
# text causes empty output (model has no "turn" to respond to) or off-task
# completion (model just continues whatever pattern it last saw).
JAIS_TEMPLATE_AR = (
    "### Instruction: اسمك \"جيس\" وسميت على اسم جبل جيس اعلى جبل في الامارات. "
    "تم بنائك بواسطة Inception في الإمارات. أنت مساعد مفيد ومحترم وصادق. "
    "أجب دائمًا بأكبر قدر ممكن من المساعدة، مع الحفاظ على البقاء أمناً. "
    "أكمل المحادثة بين [|Human|] و[|AI|] :\n"
    "### Input:[|Human|] {question}\n"
    "[|AI|]\n"
    "### Response :"
)

# 4) Engineered prompt builder — note the explicit "now analyze THIS sentence"
# cue at the very end, right before generation starts, so the model can't
# fall back to repeating the few-shot example.
def build_engineered_prompt(sentence: str) -> str:
    question = f"""### Role
أنت أستاذ متخصص في النحو العربي والإعراب، وتلتزم بقواعد النحو العربي الفصيح التقليدي.

### Context
حلّل الجملة العربية نحويًا بدقة. المطلوب هو الإعراب فقط دون شرح المعنى أو الترجمة.

### Constraints
- حدّد كل مكوّن نحوي في الجملة وضعه في صف مستقل.
- فكّك اللواصق النحوية المتصلة بالكلمات، مثل:
  - و، ف: حروف عطف.
  - ب، ك، ل: حروف جر.
  - الضمائر المتصلة: ـه، ـها، ـك، ـي، ـنا، ـهم وغيرها.
- مثال: "بالكرة" تُفكّك إلى "بـ" + "الكرة"، و"والكتاب" إلى "و" + "الكتاب".
- لا تفكك "الـ" التعريف، ولا الجذور أو الأوزان الصرفية.
- إذا كان الفاعل أو نائب الفاعل غير ظاهر وكان ضميرًا مستترًا، أضف صفًا مستقلًا بعد الفعل مباشرة بصيغة:
  `[ضمير مستتر تقديره: X]`
- لا تضف ضميرًا مستترًا إذا كان الفاعل أو نائب الفاعل ظاهرًا.
- حدّد وظيفة الضمير المتصل حسب السياق؛ فقد يكون مفعولًا به أو مضافًا إليه أو غير ذلك.
- ميّز بين المعرب والمبني، واذكر المحل الإعرابي للمبني عند الحاجة.
- حافظ على ترتيب المكونات.
- لا تستخرج الجذور أو المعاني.
- أخرج جدولًا فقط دون أي نص إضافي.

### Example
الجملة: "نـلعب بالكرة"

| الكلمة / المقطع | نوع الكلمة | الموقع الإعرابي | الحالة | العلامة وسببها |
|---|---|---|---|---|
| نـ | ضمير تقديره نحن | فاعل | مبني | في محل رفع فاعل |
| نلعب | فعل مضارع | فعل | مرفوع | الضمة الظاهرة |
| بـ | حرف جر | حرف جر | مبني | مبني على الكسر، لا محل له |
| الكرة | اسم | اسم مجرور | مجرور | الكسرة الظاهرة |

### Task
أعرب الجملة التالية فقط:
"{sentence}"

### Steps (Internal Analysis)
1. افصل اللواصق النحوية المتصلة.
2. حدّد الفاعل أو نائب الفاعل، واستخرج الضمير المستتر عند وجوده فقط.
3. حدّد نوع كل مكوّن ووظيفته.
4. حدّد الحالة الإعرابية أو البناء والعلامة.
5. راجع النتيجة ثم أخرج الجدول فقط.

### Expected Output Format
| الكلمة / المقطع | نوع الكلمة | الموقع الإعرابي | الحالة | العلامة وسببها |
|---|---|---|---|---|
"""
    return JAIS_TEMPLATE_AR.format(question=question)

# 5) Generation helper — extract text after "### Response :" exactly as the
# model card's own reference implementation does, instead of diffing decoded
# strings (which can silently break due to tokenizer/special-token mismatches
# and was the likely cause of the "empty" baseline output).
def generate_response(prompt: str, max_new_tokens: int = 400) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # deterministic output — important for grammar parsing
            temperature=0.1,
            repetition_penalty=1.2,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("### Response :")[-1].strip()

In [42]:
# 6) Test sentence: "The student read the book in the library"
sentence = "قرأ الطالبُ الكتابَ في المكتبةِ"

# Baseline must ALSO use the native template — otherwise it's not a fair
# comparison, since an un-wrapped prompt fails for a structural reason
# (wrong format) rather than a prompt-engineering reason (missing detail).
baseline_question = f"قم باعراب هذه الجملة التالية:\n\"{sentence}\""
baseline_prompt = JAIS_TEMPLATE_AR.format(question=baseline_question)
engineered_prompt = build_engineered_prompt(sentence)

baseline_output = generate_response(baseline_prompt)
engineered_output = generate_response(engineered_prompt)

print("=== Baseline prompt result ===")
print(baseline_output)

print("\n=== Engineered prompt result ===")
print(engineered_output)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (10928) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


=== Baseline prompt result ===
الجملة التي قدمتها مكتوبة بالفعل باللغة العربية بشكل صحيح وواضح. ولكن إذا كنت ترغب في إعرابها (أي تقسيمها إلى مكوناتها الأساسية)، يمكننا القول أنها تتألف من العناصر التالية:

1. "الطالب": هو الفاعل في الجملة وهو الذي يقوم بالعمل.
2. "قرأ": هذا الفعل وهو العمل الذي يقوم به الفاعل.
3. "الكتاب": هو المفعول به وهو الشيء الذي يتأثر بالفعل.
4. "في": حرف جر يدل على المكان.
5. "المكتبة": هي الاسم المجرور بسبب حرف الجر "في". وهي تدل على المكان الذي تمت فيه القراءة.

إذاً، يمكن تفسير الجملة كالتالي: الطالب قرأ كتابا في المكتبة.

=== Engineered prompt result ===
بناءً على التحليل السابق، يمكن تكوين الجدول التالي للجملة "قرأ الطالبُ الكتابَ في المكتبةِ":

| الكلمة      | نوع الكلمة | الموقع الإعرابي | الحالة    | العلامة وسببها |
|------------|-----------|--------------------|----------|---------------|
| الطالبُ   | اسم       | الفاعل             | مرفوع    | الضمة الظاهرة    |
| قرأ      | فعل ماضي | فعل               | مبني على الفتح | لا محل له من الإعراب |
| الط

## Testing with a more difficult sentence

In [43]:
from IPython.display import HTML

# --- Show the input sentence ---
display(HTML(
    "<div dir='rtl' style='text-align:right; font-size:18px; "
    "background:black; color:white; padding:10px; border-radius:6px;'>"
    f"<b>Input sentence:</b> {sentence}</div>"
))

# --- Always show the FULL, unedited response exactly as the model returned
# it, then additionally render a parsed Markdown table below it if one is
# present in the text (without dropping any surrounding text) ---
import re

def show_full_response(title: str, text: str):
    display(HTML(f"<h3 dir='rtl' style='text-align:right;'>{title}</h3>"))

    body = text.strip()
    if not body:
        display(HTML(
            "<p dir='rtl' style='text-align:right; color:white; background:black; "
            "padding:10px; border-radius:6px;'>*(completely empty output)*</p>"
        ))
        return

    # 1) Always show the full raw text — nothing dropped or truncated
    display(HTML(
        "<pre dir='rtl' style='text-align:right; direction:rtl; white-space:pre-wrap; "
        "font-family:inherit; background:black; color:white; border:1px solid #ddd; "
        "padding:10px; border-radius:6px; font-size:15px;'>"
        f"{body}</pre>"
    ))

    # 2) If a Markdown table is present anywhere in the text, additionally
    # render it as a formatted RTL HTML table
    if "|" in body:
        # Keep any text before the table instead of discarding it; only
        # operate on the substring starting at the first "|"
        table_part = "|" + body.split("|", 1)[-1]
        cleaned = re.sub(r"\s*\|\s*(?=\|[^|]*\|)", "\n|", table_part)
        rows = [r.strip() for r in cleaned.split("\n") if r.strip().startswith("|")]

        if rows:
            html_rows = []
            for i, row in enumerate(rows):
                cells = [c.strip() for c in row.strip("|").split("|")]
                if all(set(c) <= set("-: ") for c in cells):
                    continue  # skip the separator row, e.g. |---|---|
                tag = "th" if i == 0 else "td"
                cells_html = "".join(
                    f"<{tag} style='border:1px solid #555; padding:6px; color:white;'>{c}</{tag}>"
                    for c in cells
                )
                html_rows.append(f"<tr>{cells_html}</tr>")

            if html_rows:
                table_html = (
                    "<table dir='rtl' style='direction:rtl; text-align:right; "
                    "border-collapse:collapse; width:100%; font-size:15px; "
                    "background:black; color:white; margin-top:8px;'>"
                    + "".join(html_rows) +
                    "</table>"
                )
                display(HTML(table_html))

show_full_response("Full raw response — baseline prompt", baseline_output)
show_full_response("Full raw response — engineered prompt", engineered_output)

الكلمة,نوع الكلمة,الموقع الإعرابي,الحالة,العلامة وسببها
الطالبُ,اسم,الفاعل,مرفوع,الضمة الظاهرة
قرأ,فعل ماضي,فعل,مبني على الفتح,لا محل له من الإعراب
الطالبُ,اسم,الفاعل,مرفوع,الضمة الظاهرة
الكتابَ,اسم,المفعول به,مجرور,الكسرة الظاهرة
في,حرف جر,حرف,مبني,لا محل له من الإعراب
المكتبةِ,اسم,"المجرور بـ ""في""",مجرور,الكسرة الظاهرة


In [117]:
# A more complex sentence for testing
sentence = "نكتب بـالقلم"

# Re-generate prompts with the new sentence
baseline_question = f"قم باعراب هذه الجملة التالية:\n\"{sentence}\""
baseline_prompt = JAIS_TEMPLATE_AR.format(question=baseline_question)
engineered_prompt = build_engineered_prompt(sentence)

# Generate responses
baseline_output = generate_response(baseline_prompt)
engineered_output = generate_response(engineered_prompt)

# Display results using the defined function
show_full_response("Full raw response — baseline prompt (Difficult Sentence)", baseline_output)
show_full_response("Full raw response — engineered prompt (Difficult Sentence)", engineered_output)

الكلمة / المقطع,نوع الكلمة,الموقع الإعرابي,الحالة,العلامة وسببها
ن,ضمير متصل,فاعل,مبني,في محل رفع فاعل
أكتب,فعل مضارع,فعل,مرفوع,الضمة الظاهرة
بـ,حرف جر,حرف جر,مبني,مبني على الفتح، لا محل له
القلم,اسم,اسم مجرور,مجرور,الكسرة الظاهرة
